In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score
import numpy as np
import os
os.chdir('/media/my_drives')

In [ ]:
df = pd.read_csv('Results.csv')
df.tail()

,Model,Dataset,N_Training_Samples,BATCH_SIZE,LEARNING_RATE,EPOCHS,Time_Seconds,Accuracy,BIGRAM_BOOL,BINARY_BOOL
3775,Ensemble_AllProba,FashionImages,200.0,32.0,0.00100,32.0,0.35,0.915698,NaN,NaN
3776,Ensemble_AllProba,FashionImages,200.0,32.0,0.00001,32.0,0.36,0.363372,NaN,NaN
3777,Ensemble_AllProba,FashionImages,200.0,64.0,0.01000,20.0,0.15,0.927326,NaN,NaN
3778,Ensemble_AllProba,FashionImages,200.0,64.0,0.00100,32.0,0.23,0.773256,NaN,NaN
3779,Ensemble_AllProba,FashionImages,200.0,64.0,0.00001,32.0,0.23,0.098837,NaN,NaN


In [3]:
df_agg = df.loc[df.groupby(['Model', 'Dataset', 'N_Training_Samples'])['Accuracy'].idxmax()]
df_agg = df_agg.sort_values(['N_Training_Samples', 'Dataset', 'Accuracy'])
df_agg = df_agg.reset_index(drop=True)
datasets = df_agg.Dataset.unique()
df_agg.tail()

,Model,Dataset,N_Training_Samples,BATCH_SIZE,LEARNING_RATE,EPOCHS,Time_Seconds,Accuracy,BIGRAM_BOOL,BINARY_BOOL
310,Inception,sentiment,1000.0,64.0,0.01000,6.0,102.4900,0.604457,NaN,NaN
311,ViT,sentiment,1000.0,16.0,0.00001,8.0,154.2813,0.607242,NaN,NaN
312,ResNet,sentiment,1000.0,32.0,0.00100,2.0,79.3794,0.615599,NaN,NaN
313,Ensemble_Descriptions,sentiment,1000.0,16.0,0.00100,13.0,1.8100,0.662953,False,True
314,Ensemble_AllProba,sentiment,1000.0,16.0,0.01000,28.0,1.8600,0.668524,NaN,NaN


In [ ]:
"""
Add zero_few-shot
"""

def is_actual_prediction(prediction: str):
    if any(word in prediction.lower() for word in ['sorry', 'cant']):
        return np.nan
    return prediction

MODEL_NAME = 'Phi' # GPT, Phi
PRED_COLUMN = 'prediction_phi4_combined' #gpt5, phi4

for dataset in datasets:
    df = pd.read_csv(f'CSV_Holdout/{dataset}.csv')
    df = df.apply(lambda x: x.str.lower() if x.dtype == "object" else x)
    df[PRED_COLUMN] = df[PRED_COLUMN].apply(is_actual_prediction)
    df = df.dropna(subset=[PRED_COLUMN])
    
    accuracy = accuracy_score(df['label'], df[PRED_COLUMN]) 
    for N_Samples in [200, 1000]:
        if (dataset == 'AIDA') & (N_Samples == 1000):
            continue
        counter = df_agg.shape[0]
        df_agg.at[counter, 'Model'] = MODEL_NAME
        df_agg.at[counter, 'Dataset'] = dataset
        df_agg.at[counter, 'N_Training_Samples'] = N_Samples
        df_agg.at[counter, 'Accuracy'] = accuracy

MODEL_NAME = 'GPT' # GPT, Phi
PRED_COLUMN = 'prediction_gpt5_combined' #gpt5, phi4

for dataset in datasets:
    df = pd.read_csv(f'CSV_Holdout/{dataset}.csv')
    df = df.apply(lambda x: x.str.lower() if x.dtype == "object" else x)
    df[PRED_COLUMN] = df[PRED_COLUMN].apply(is_actual_prediction)
    df = df.dropna(subset=[PRED_COLUMN])
    
    accuracy = accuracy_score(df['label'], df[PRED_COLUMN]) 
    for N_Samples in [200, 1000]:
        if (dataset == 'AIDA') & (N_Samples == 1000):
            continue
        counter = df_agg.shape[0]
        df_agg.at[counter, 'Model'] = MODEL_NAME
        df_agg.at[counter, 'Dataset'] = dataset
        df_agg.at[counter, 'N_Training_Samples'] = N_Samples
        df_agg.at[counter, 'Accuracy'] = accuracy

In [5]:
df_agg = df_agg.sort_values(by=['N_Training_Samples', 'Dataset', 'Accuracy'], ascending=[False, True, False])
df_agg

,Model,Dataset,N_Training_Samples,BATCH_SIZE,LEARNING_RATE,EPOCHS,Time_Seconds,Accuracy,BIGRAM_BOOL,BINARY_BOOL
170,Ensemble_AllProba,Brand-Styles,1000.0,32.0,0.01000,11.0,1.5700,0.949861,NaN,NaN
169,Ensemble_Descriptions,Brand-Styles,1000.0,16.0,0.00100,15.0,1.4700,0.935933,True,True
168,ConvNeXtBase,Brand-Styles,1000.0,16.0,0.00001,19.0,608.8515,0.927577,NaN,NaN
167,ViT,Brand-Styles,1000.0,16.0,0.00001,32.0,381.8334,0.874652,NaN,NaN
352,GPT,Brand-Styles,1000.0,NaN,NaN,NaN,NaN,0.841226,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
154,ConvNeXtBase,sentiment,200.0,16.0,0.01000,1.0,64.6810,0.587744,NaN,NaN
155,Full-NN-Training,sentiment,200.0,16.0,0.01000,1.0,28.2300,0.587744,NaN,NaN
156,ViT,sentiment,200.0,16.0,0.01000,11.0,106.5630,0.587744,NaN,NaN
153,Xception,sentiment,200.0,16.0,0.00100,8.0,108.0000,0.584958,NaN,NaN


In [ ]:
df_agg.to_csv('Results_Filtered.csv', index=False)